In [0]:
%sql
USE CATALOG ecommerce_logistics;
USE SCHEMA logistics

In [0]:
df=spark.table("bronze_geolocation")
df.show(100)

In [0]:
from pyspark.sql.functions import col, lower, trim, regexp_replace,concat

In [0]:
df.groupBy("geolocation_zip_code_prefix").count().show()

In [0]:
%sql
DESCRIBE TABLE bronze_geolocation

In [0]:
df.filter(col("geolocation_zip_code_prefix")==1046).show()

In [0]:
df.filter(col("geolocation_city")=="sp").show()

In [0]:
df.groupBy(["geolocation_city"]).count().show(50,truncate=False)

In [0]:
mapping = {
    "sp": "sao paulo",
    "sa£o paulo": "sao paulo",
    "sãopaulo": "sao paulo",
    "sbcampo": "sao bernardo do campo",
    "mogidascruzes": "mogi das cruzes",
    "belo horizonta": "belo horizonte",
    "bh": "belo horizonte",
    "embu guacu": "embu-guacu",
    "embuguacu": "embu-guacu",
    "biritiba mirim": "biritiba-mirim",
    "sao luiz do paraitinga": "sao luis do paraitinga",
    "alta floresta do oeste": "alta floresta d'oeste",
    "alta floresta doeste": "alta floresta d'oeste",
    "aparecida d oeste": "aparecida d'oeste",
    "aparecida doeste": "aparecida d'oeste",
    "arraial d ajuda": "arraial d'ajuda",
    "armacao de buzios": "armacao dos buzios",
    "buzios": "armacao dos buzios",
    "balneario de picarras": "balneario picarras",
    "bataipora": "bataypora",
    "barrado quarai": "barra do quarai",
    "4º centenario": "4o. centenario",
    "bacaxa (saquarema) - distrito": "bacaxa",
    "california da barra (barra do pirai)": "california da barra",
    "...arraial do cabo": "arraial do cabo",
    "aquidaban": "aquidaba",
    "arenopolis": "arenapolis",
    "brazopolis": "brasopolis",
    "cachoeira de piria": "cachoeira do piria"
}

In [0]:
silver_geolocation = (
    spark.table("bronze_geolocation")
    .withColumn(
        "geolocation_city",
        lower(trim(col("geolocation_city")))
    )
    .withColumn(
        "geolocation_city",
        regexp_replace(col("geolocation_city"), "[áàâãä]", "a")
    )
    .withColumn(
        "geolocation_city",
        regexp_replace(col("geolocation_city"), "[éèêë]", "e")
    )
    .withColumn(
        "geolocation_city",
        regexp_replace(col("geolocation_city"), "[íìîï]", "i")
    )
    .withColumn(
        "geolocation_city",
        regexp_replace(col("geolocation_city"), "[óòôõö]", "o")
    )
    .withColumn(
        "geolocation_city",
        regexp_replace(col("geolocation_city"), "[úùûü]", "u")
    )
    .withColumn(
        "geolocation_city",
        regexp_replace(col("geolocation_city"), "ç", "c")
    )
    .withColumn(
        "geolocation_city",
        regexp_replace(col("geolocation_city"), "ñ", "n")
    )
    .withColumn(
        "geolocation_city",
        regexp_replace(col("geolocation_city"), r"\s+", " ")
    ).replace(mapping,subset=["geolocation_city"])
)

In [0]:
silver_geolocation.groupBy(["geolocation_city"]).count().orderBy("geolocation_city").show(3000, truncate=False)

In [0]:
silver_geolocation.withColumn("KEY",concat(col("geolocation_zip_code_prefix"),col("geolocation_lat"),col("geolocation_lng"),col("geolocation_city"),col("geolocation_state"))).dropDuplicates(["KEY"]).drop("KEY").withColumn("geolocation_zip_code_prefix",col("geolocation_zip_code_prefix").cast("string")).write.mode("overwrite").saveAsTable("silver_geolocation")

In [0]:
silver_geolocation=spark.table("silver_geolocation").show(50,truncate=False)

In [0]:
bronze_sellers=spark.table("bronze_sellers")

In [0]:
%sql
DESCRIBE TABLE bronze_sellers

In [0]:
city_mapping = {
    "sao pauo": "sao paulo",
    "sao paluo": "sao paulo",
    "sao paulop": "sao paulo",
    "sao  paulo": "sao paulo",
    "são paulo": "sao paulo",
    "sao paulo sp": "sao paulo",
    "sao paulo - sp": "sao paulo",
    "sao paulo / sao paulo": "sao paulo",
    "sp / sp": "sao paulo",
    "sao bernardo do capo": "sao bernardo do campo",
    "ao bernardo do campo": "sao bernardo do campo",
    "sbc": "sao bernardo do campo",
    "sbc/sp": "sao bernardo do campo",
    "ribeirao pretp": "ribeirao preto",
    "riberao preto": "ribeirao preto",
    "robeirao preto": "ribeirao preto",
    "ribeirao preto / sao paulo": "ribeirao preto",
    "sando andre": "santo andre",
    "santo andre/sao paulo": "santo andre",
    "sao jose dos pinhas": "sao jose dos pinhais",
    "sao  jose dos pinhais": "sao jose dos pinhais",
    "sao jose do rio pret": "sao jose do rio preto",
    "s jose do rio preto": "sao jose do rio preto",
    "scao jose do rio pardo": "sao jose do rio pardo",
    "mogi das cruses": "mogi das cruzes",
    "mogi das cruzes / sp": "mogi das cruzes",
    "rio de janeiro / rio de janeiro": "rio de janeiro",
    "rio de janeiro \\rio de janeiro": "rio de janeiro",
    "rio de janeiro, rio de janeiro, brasil": "rio de janeiro",
    "balenario camboriu": "balneario camboriu",
    "belo horizont": "belo horizonte",
    "floranopolis": "florianopolis",
    "garulhos": "guarulhos",
    "juzeiro do norte": "juazeiro do norte",
    "cascavael": "cascavel",
    "ferraz de  vasconcelos": "ferraz de vasconcelos",
    "paincandu": "paicandu",
    "portoferreira": "porto ferreira",
    "auriflama/sp": "auriflama",
    "lages - sc": "lages",
    "pinhais/pr": "pinhais",
    "andira-pr": "andira",
    "sao sebastiao da grama/sp": "sao sebastiao da grama",
    "angra dos reis rj": "angra dos reis",
    "maua/sao paulo": "maua",
    "cariacica / es": "cariacica",
    "jacarei / sao paulo": "jacarei",
    "barbacena/ minas gerais": "barbacena",
    "carapicuiba / sao paulo": "carapicuiba",
    "novo hamburgo, rio grande do sul, brasil": "novo hamburgo",
    "sao miguel d'oeste": "sao miguel do oeste",
    "santa barbara d oeste": "santa barbara d'oeste",
    "santa barbara d´oeste": "santa barbara d'oeste",
    "arraial d'ajuda (porto seguro)": "arraial d'ajuda",
    "brasilia df": "brasilia",
    "aguas claras df": "aguas claras",
    "ji parana": "ji-parana",
    "tabao da serra": "taboao da serra"
}

In [0]:
bronze_sellers.replace(city_mapping,subset="seller_city").groupBy("seller_city").count().orderBy("count").show(1000,truncate=False)

In [0]:
bronze_sellers.groupBy("seller_zip_code_prefix").count().orderBy("count").show(1000,truncate=False)

In [0]:
bronze_sellers.withColumn("KEY",concat(col("seller_id"),col("seller_zip_code_prefix"),col("seller_city"),col("seller_state"))).groupBy("KEY").count().orderBy(col("count").desc()).show(100)

In [0]:
bronze_sellers.withColumn("seller_zip_code_prefix",col("seller_zip_code_prefix").cast("string")).replace(city_mapping,subset="seller_city").write.format("delta").mode("overwrite").saveAsTable("silver_sellers")